In [2]:
print("Hello! The robot is awake!")


Hello! The robot is awake!


In [3]:
# This is like buying the tools for our workshop
!pip install requests pandas crewai langchain_groq

In [4]:
!pip install --user requests pandas crewai langchain_groq

In [5]:
!pip install --user requests pandas crewai langchain_groq

In [6]:
!pip install --user requests pandas crewai langchain_groq

In [7]:
import requests
import pandas as pd
from groq import Groq

print("🎉 The tools are ready in our workshop!")

🎉 The tools are ready in our workshop!


In [8]:
!%pip install requests pandas groq crewai langchain_groq


'%pip' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
!pip install requests pandas groq crewai langchain_groq

In [3]:
import pandas as pd
import requests
from groq import Groq
import streamlit as st

print("🚀 BOOM! The workshop is open and every tool is working!")

🚀 BOOM! The workshop is open and every tool is working!


In [5]:
import pandas as pd
import requests

# This is where we store our secret keys (like a safe)
ALPHA_VANTAGE_KEY = '********************************************************'
GROQ_API_KEY = '*************************************************************'

# Let's pick a stock to test, like NVIDIA (NVDA) or Apple (AAPL)
SYMBOL = 'AAPL'

def get_stock_data(symbol):
    # This is the URL address to Alpha Vantage's library
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
    
    # We knock on the door and ask for data
    response = requests.get(url)
    data = response.json()
    
    # We take the "Daily Time Series" part of the data
    df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index')
    
    # We clean the data so it's easy for our bot to read
    df = df.astype(float)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index() # Make sure dates are in order from old to new
    
    return df

# Let's try to fetch the data!
try:
    prices = get_stock_data(SYMBOL)
    print(f"✅ Success! We grabbed the last {len(prices)} days of data for {SYMBOL}.")
    print(prices.tail(5)) # This shows us the last 5 rows
except Exception as e:
    print(f"❌ Oops! Something went wrong: {e}")

✅ Success! We grabbed the last 100 days of data for AAPL.
            1. open  2. high  3. low  4. close   5. volume
2026-05-07  289.270   292.13  285.78    287.44  45224300.0
2026-05-08  290.010   294.76  290.00    293.32  52692761.0
2026-05-11  291.979   293.88  290.23    292.68  42247285.0
2026-05-12  292.560   295.27  292.56    294.80  45748129.0
2026-05-13  293.500   300.92  293.50    298.87  52684260.0


In [12]:
def calculate_signals(df):
    # 1. Calculate the moving averages
    df['MA10'] = df['4. close'].rolling(window=10).mean()
    df['MA30'] = df['4. close'].rolling(window=30).mean()
    
    # 2. Find the Signal
    # If Rabbit > Turtle, Signal is 1 (BUY). Otherwise, it's 0 (SELL/WAIT)
    df['Signal'] = 0.0
    df.loc[df['MA10'] > df['MA30'], 'Signal'] = 1.0
    
    # 3. Find the "Crossover" moment
    # This tells us exactly when the lines cross each other
    df['Crossover'] = df['Signal'].diff()
    
    return df

# Apply the math to our prices
prices_with_signals = calculate_signals(prices)

# Let's see the latest math!
print("📈 Math Complete! Checking the latest 5 days for signals:")
print(prices_with_signals[['4. close', 'MA10', 'MA30', 'Signal']].tail(5))

📈 Math Complete! Checking the latest 5 days for signals:
            4. close     MA10        MA30  Signal
2026-05-07    287.44  276.700  265.893333     1.0
2026-05-08    293.32  278.926  267.241000     1.0
2026-05-11    292.68  281.433  268.703667     1.0
2026-05-12    294.80  283.842  270.309333     1.0
2026-05-13    298.87  286.712  271.812000     1.0


In [13]:
!pip install litellm

In [6]:
import os
from crewai import Agent, Task, Crew, LLM

# 1. Ensure the key is set
os.environ["GROQ_API_KEY"] = "*************************************************"
# 2. Setup the NEW Brain
# We are changing the model name to one that is currently active
my_ai_brain = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

# 3. Define the Agent
analyst_agent = Agent(
    role='Senior Trading Advisor',
    goal='Explain stock signals simply',
    backstory='You are a friendly pro trader who loves teaching beginners.',
    llm=my_ai_brain,
    verbose=True,
    allow_delegation=False
)

# 4. Grab our data (make sure you ran the cells above first!)
latest_price = prices_with_signals['4. close'].iloc[-1]
current_signal = "BUY" if prices_with_signals['Signal'].iloc[-1] == 1.0 else "SELL"

# 5. The Job
analysis_job = Task(
    description=f"Apple (AAPL) is at ${latest_price}. The signal is {current_signal}. Tell the user why this is happening in 2 simple sentences.",
    expected_output="A very simple 2-sentence trading explanation.",
    agent=analyst_agent
)

# 6. Start the Crew!
my_crew = Crew(agents=[analyst_agent], tasks=[analysis_job])
report = my_crew.kickoff()

print("\n\n" + "="*30)
print("🤖 AI TRADING REPORT:")
print(report)
print("="*30)

NameError: name 'prices_with_signals' is not defined

In [16]:
!pip install streamlit

In [8]:
%%writefile Quantora.py
import streamlit as st
import pandas as pd
import requests
import os
from crewai import Agent, Task, Crew, LLM

# --- CONFIGURATION ---
st.set_page_config(page_title="Quantora AI", page_icon="📈", layout="wide")

# Secret Keys (In a real app, we use 'st.secrets', but for now we'll use these)
ALPHA_VANTAGE_KEY = 'YOUR_ALPHA_VANTAGE_KEY'
os.environ["GROQ_API_KEY"] = "************************************"

# --- APP UI ---
st.title("🏹 Quantora")
st.markdown("### *AI-Powered Quantitative Stock Signals*")
st.divider()

symbol = st.text_input("Enter Ticker Symbol:", "AAPL").upper()

if st.button("Generate Signal Report"):
    with st.spinner(f"Quantora is analyzing {symbol}..."):
        try:
            # 1. Fetch Data
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
            response = requests.get(url)
            data = response.json()
            df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index').astype(float)
            df.index = pd.to_datetime(df.index)
            df = df.sort_index()

            # 2. Calculate Math
            df['MA10'] = df['4. close'].rolling(window=10).mean()
            df['MA30'] = df['4. close'].rolling(window=30).mean()
            current_price = df['4. close'].iloc[-1]
            signal = "BUY" if df['MA10'].iloc[-1] > df['MA30'].iloc[-1] else "SELL"

            # 3. AI Analysis
            my_ai_brain = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
            
            analyst = Agent(
                role='Senior Quant Researcher',
                goal=f'Explain why {symbol} has a {signal} signal.',
                backstory='You provide elite institutional-grade analysis.',
                llm=my_ai_brain,
                allow_delegation=False
            )

            job = Task(
                description=f"Price: ${current_price}. Signal: {signal}. Explain in 2 sentences for a premium subscriber.",
                expected_output="A professional 2-sentence summary.",
                agent=analyst
            )

            report = Crew(agents=[analyst], tasks=[job]).kickoff()

            # 4. Display Results
            col1, col2 = st.columns(2)
            col1.metric("Current Price", f"${current_price}")
            col2.metric("Signal", signal)
            
            st.success("**Quantora Intelligence Report:**")
            st.write(report.raw)
            
        except Exception as e:
            st.error(f"Quantora encountered an error: {e}")

Overwriting Quantora.py


In [18]:
!pip install plotly

In [7]:
%%writefile Quantora.py
import streamlit as st
import pandas as pd
import requests
import os
import plotly.graph_objects as go
from crewai import Agent, Task, Crew, LLM

# --- CONFIGURATION ---
st.set_page_config(page_title="Quantora AI", page_icon="📈", layout="wide")

ALPHA_VANTAGE_KEY = 'YOUR_KEY_HERE' # <--- Remember to put your key back!
os.environ["GROQ_API_KEY"] = "***************************************************"

# --- APP UI ---
st.title("🏹 Quantora")
st.markdown("### *AI-Powered Quantitative Stock Signals*")
st.divider()

symbol = st.text_input("Enter Ticker Symbol:", "NVDA").upper()

if st.button("Generate Signal Report"):
    with st.spinner(f"Quantora is analyzing {symbol}..."):
        try:
            # 1. Fetch Data
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
            response = requests.get(url)
            data = response.json()
            df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index').astype(float)
            df.index = pd.to_datetime(df.index)
            df = df.sort_index().tail(60) # Look at the last 60 days

            # 2. Math
            df['MA10'] = df['4. close'].rolling(window=10).mean()
            df['MA30'] = df['4. close'].rolling(window=30).mean()
            current_price = df['4. close'].iloc[-1]
            signal = "BUY" if df['MA10'].iloc[-1] > df['MA30'].iloc[-1] else "SELL"

            # 3. VISUAL ANALYSIS (New Section!)
            fig = go.Figure()
            # Actual Price
            fig.add_trace(go.Scatter(x=df.index, y=df['4. close'], name='Price', line=dict(color='white', width=2)))
            # Fast Moving Average (Rabbit)
            fig.add_trace(go.Scatter(x=df.index, y=df['MA10'], name='MA10 (Fast)', line=dict(color='cyan', dash='dot')))
            # Slow Moving Average (Turtle)
            fig.add_trace(go.Scatter(x=df.index, y=df['MA30'], name='MA30 (Slow)', line=dict(color='orange')))
            
            fig.update_layout(title=f"{symbol} Price Trend", template="plotly_dark", xaxis_title="Date", yaxis_title="Price ($)")
            st.plotly_chart(fig, use_container_width=True)

            # 4. AI Analysis
            my_ai_brain = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
            analyst = Agent(role='Senior Quant Researcher', goal=f'Explain {symbol} signal.', backstory='Elite analyst.', llm=my_ai_brain)
            job = Task(description=f"Price: {current_price}. Signal: {signal}. Explain in 2 sentences.", expected_output="Summary.", agent=analyst)
            report = Crew(agents=[analyst], tasks=[job]).kickoff()

            # 5. Display Results
            col1, col2 = st.columns(2)
            col1.metric("Current Price", f"${current_price}")
            col2.metric("Signal", signal, delta="Positive Trend" if signal == "BUY" else "Negative Trend")
            
            st.success("**Quantora Intelligence Report:**")
            st.write(report.raw)
            
        except Exception as e:
            st.error(f"Quantora encountered an error: {e}")

Overwriting Quantora.py


In [21]:
%%writefile Quantora.py
import streamlit as st
import pandas as pd
import requests
import os
import smtplib
from email.message import EmailMessage
import plotly.graph_objects as go
from crewai import Agent, Task, Crew, LLM

# --- CONFIGURATION ---
st.set_page_config(page_title="Quantora AI", page_icon="📈", layout="wide")

ALPHA_VANTAGE_KEY = 'YOUR_KEY_HERE' 
os.environ["GROQ_API_KEY"] = "***************************************************"

# --- EMAIL FUNCTION ---
def send_alert(target_email, stock, signal_type):
    try:
        msg = EmailMessage()
        msg.set_content(f"Quantora Alert: {stock} has triggered a {signal_type} signal!")
        msg['Subject'] = f"🚀 Quantora Trade Alert: {stock}"
        msg['From'] = "alerts@quantora.ai" # This can be anything
        msg['To'] = target_email

        # NOTE: To actually send, you'd need an SMTP server (like Gmail)
        # For this demo, we will log it to the screen to show it's working
        return True
    except:
        return False

# --- APP UI ---
st.sidebar.title("📧 Alert Settings")
user_email = st.sidebar.text_input("Your Email for Alerts:")
enable_alerts = st.sidebar.checkbox("Enable Real-Time Alerts")

st.title("🏹 Quantora")
st.markdown("### *AI-Powered Quantitative Stock Signals*")
st.divider()

symbol = st.text_input("Enter Ticker Symbol:", "NVDA").upper()

if st.button("Generate Signal Report"):
    with st.spinner(f"Quantora is analyzing {symbol}..."):
        try:
            # 1. Fetch Data
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
            response = requests.get(url)
            data = response.json()
            df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index').astype(float)
            df.index = pd.to_datetime(df.index)
            df = df.sort_index().tail(60)

            # 2. Math
            df['MA10'] = df['4. close'].rolling(window=10).mean()
            df['MA30'] = df['4. close'].rolling(window=30).mean()
            current_price = df['4. close'].iloc[-1]
            signal = "BUY" if df['MA10'].iloc[-1] > df['MA30'].iloc[-1] else "SELL"

            # 3. Visuals
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=df.index, y=df['4. close'], name='Price', line=dict(color='white', width=2)))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA10'], name='MA10', line=dict(color='cyan')))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA30'], name='MA30', line=dict(color='orange')))
            fig.update_layout(template="plotly_dark")
            st.plotly_chart(fig, use_container_width=True)

            # 4. Display Results
            col1, col2 = st.columns(2)
            col1.metric("Current Price", f"${current_price}")
            col2.metric("Signal", signal)

            # 5. Handle Alert Logic
            if enable_alerts and user_email:
                st.sidebar.success(f"Alert sent to {user_email}!")
                send_alert(user_email, symbol, signal)

            # 6. AI Agent
            my_ai_brain = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
            analyst = Agent(role='Senior Quant', goal='Explain signal.', backstory='Elite.', llm=my_ai_brain)
            job = Task(description=f"Analyze {symbol} {signal} at {current_price}.", expected_output="2 sentence report.", agent=analyst)
            report = Crew(agents=[analyst], tasks=[job]).kickoff()
            
            st.success("**Quantora Intelligence Report:**")
            st.write(report.raw)
            
        except Exception as e:
            st.error(f"Error: {e}")

Overwriting Quantora.py


In [26]:
%%writefile Quantora.py
import streamlit as st
import pandas as pd
import requests
import os
import plotly.graph_objects as go
from crewai import Agent, Task, Crew, LLM

# --- THEMES & UI STYLE ---
st.set_page_config(page_title="Quantora OS", layout="wide", page_icon="🏹")
 # High-Tech CSS Injection
st.markdown("""
    <style>
    /* Main Background */
    .stApp {
        background: linear-gradient(145deg, #0d1117, #161b22);
    }
    
    /* Glassmorphism Cards */
    div[data-testid="stMetricValue"] {
        background: rgba(255, 255, 255, 0.03);
        border: 1px solid rgba(255, 255, 255, 0.1);
        padding: 15px;
        border-radius: 12px;
        backdrop-filter: blur(10px);
        color: #58a6ff !important;
        text-shadow: 0 0 10px rgba(88, 166, 255, 0.5);
    }

    /* Neon Sidebar */
    section[data-testid="stSidebar"] {
        background-color: rgba(22, 27, 34, 0.8);
        border-right: 1px solid #30363d;
    }

    /* Glowing Title */
    .title-text {
        font-family: 'Courier New', monospace;
        color: #79c0ff;
        text-align: center;
        letter-spacing: 5px;
        text-shadow: 0 0 20px #58a6ff;
    }
    </style>
    """, unsafe_allow_html=True) # <--- Fixed parameter name here

# And fixed here for the title:
st.markdown('<h1 class="title-text">QUANTORA OS v1.0</h1>', unsafe_allow_html=True)
# --- CONFIGURATION ---
ALPHA_VANTAGE_KEY = 'YP2FZZBORC62A296' 
os.environ["GROQ_API_KEY"] = "*************************************************************************"
TELEGRAM_TOKEN = '******************************************' # <-- From BotFather
TELEGRAM_CHAT_ID = '****************************************' # <-- From UserInfoBot

def send_telegram_alert(msg):
    if TELEGRAM_TOKEN != 'YOUR_TELEGRAM_TOKEN':
        url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
        params = {"chat_id": TELEGRAM_CHAT_ID, "text": msg, "parse_mode": "Markdown"}
        requests.get(url)

# --- UI COMMAND CENTER ---
st.markdown('<h1 class="title-text">QUANTORA OS v1.0</h1>', unsafe_allow_html=True)
st.divider()

with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/2592/2592201.png", width=80)
    st.title("System Controls")
    enable_tg = st.toggle("Telegram Direct Uplink", value=True)
    st.info("Status: Online & Encrypted")

symbol = st.text_input("INPUT TICKER SYMBOL:", "NVDA").upper()

if st.button("EXECUTE ANALYSIS SCAN"):
    with st.spinner("Analyzing Market Vectors..."):
        try:
            # 1. DATA
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
            data = requests.get(url).json()
            df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index').astype(float).sort_index().tail(60)

            # 2. CALCS
            df['MA10'] = df['4. close'].rolling(window=10).mean()
            df['MA30'] = df['4. close'].rolling(window=30).mean()
            current_price = df['4. close'].iloc[-1]
            signal = "BUY" if df['MA10'].iloc[-1] > df['MA30'].iloc[-1] else "SELL"

            # 3. GLOWING CHART
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=df.index, y=df['4. close'], name='Price', line=dict(color='#f0f6fc', width=3)))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA10'], name='FAST VECTOR', line=dict(color='#3fb950', dash='dot')))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA30'], name='SLOW VECTOR', line=dict(color='#f85149')))
            fig.update_layout(template="plotly_dark", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            st.plotly_chart(fig, use_container_width=True)

            # 4. METRICS
            c1, c2 = st.columns(2)
            c1.metric("CURRENT VALUE", f"${current_price}")
            c2.metric("SIGNAL STATUS", signal)

            # 5. TELEGRAM UPLINK
            if enable_tg:
                alert_text = f"🏹 *QUANTORA SCAN COMPLETE*\n\nTicker: `{symbol}`\nSignal: *{signal}*\nPrice: `${current_price}`"
                send_telegram_alert(alert_text)
                st.sidebar.success("Telegram Alert Dispatched")

            # 6. AI INTEL
            my_ai_brain = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
            analyst = Agent(role='Quant Specialist', goal='Provide intel.', backstory='Expert analyst.', llm=my_ai_brain)
            task = Task(description=f"Analyze {symbol} {signal} at {current_price}.", expected_output="2 sentence report.", agent=analyst)
            report = Crew(agents=[analyst], tasks=[task]).kickoff()
            
            st.markdown("### 🤖 Intelligence Briefing")
            st.info(report.raw)

        except Exception as e:
            st.error(f"System Error: {e}")

Overwriting Quantora.py


In [1]:
%%writefile Quantora.py
import streamlit as st
import pandas as pd
import requests
import os
import smtplib
from email.message import EmailMessage
import plotly.graph_objects as go
from crewai import Agent, Task, Crew, LLM

# --- THEME CONFIGURATION ---
st.set_page_config(page_title="Quantora Pro", layout="wide", page_icon="🏹")

# FinTech UI CSS Injection
st.markdown("""
    <style>
    /* Main App Background - Clean Slate */
    .stApp { background-color: #F4F7F9; }
    
    /* Card Styling (Bento Box) */
    div[data-testid="stMetricValue"] {
        background-color: #FFFFFF;
        border-radius: 16px;
        padding: 24px !important;
        box-shadow: 0 10px 25px rgba(0,0,0,0.05);
        border: 1px solid #E1E8ED;
        color: #0066FF !important;
        font-weight: 700;
    }

    /* Sidebar - Clean & Professional */
    section[data-testid="stSidebar"] {
        background-color: #FFFFFF;
        border-right: 1px solid #E1E8ED;
    }

    /* Gradient Title */
    .main-title {
        background: linear-gradient(90deg, #0066FF, #00D084);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-size: 3rem;
        font-weight: 800;
        text-align: center;
        margin-bottom: 20px;
    }

    /* Info Box Styling */
    .stAlert {
        border-radius: 12px;
        border: none;
        box-shadow: 0 4px 12px rgba(0,0,0,0.03);
    }
    </style>
    """, unsafe_allow_html=True)

# --- NOTIFICATION LOGIC ---
# --- UPDATED NOTIFICATION LOGIC ---
def send_telegram(msg, token, chat_id):
    if token and chat_id:
        try:
            # We add 'parse_mode=Markdown' so your alerts look professional with bold text
            url = f"https://api.telegram.org/bot{token}/sendMessage"
            params = {"chat_id": chat_id, "text": msg, "parse_mode": "Markdown"}
            requests.get(url, params=params)
        except Exception as e:
            st.sidebar.error(f"Telegram failed: {e}")

def send_email(msg_content, recipient):
    if recipient:
        try:
            # For Gmail, you MUST use an "App Password," not your regular password
            sender_email = "YOUR_EMAIL@gmail.com"
            sender_password = "YOUR_APP_PASSWORD" 

            msg = EmailMessage()
            msg.set_content(msg_content)
            msg['Subject'] = "🏹 Quantora Pro: Trade Signal Detected"
            msg['From'] = f"Quantora Alerts <{sender_email}>"
            msg['To'] = recipient

            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
                smtp.login(sender_email, sender_password)
                smtp.send_message(msg)
            return True
        except Exception as e:
            # This shows the error in the sidebar so it doesn't ruin your main dashboard vibe
            st.sidebar.error(f"Email system offline: {e}")
            return False
# --- CONFIGURATION ---
ALPHA_VANTAGE_KEY = '***********************' 
os.environ["GROQ_API_KEY"] = "**********************************************************"
TELEGRAM_TOKEN = '*********************************************' # <-- From BotFather
TELEGRAM_CHAT_ID = '*******************************' # <-- From UserInfoBot


# --- UI LAYOUT ---
st.markdown('<h1 class="main-title">Quantora Pro</h1>', unsafe_allow_html=True)

with st.sidebar:
    st.markdown("### 🏹 Quantora Control Center")
    st.info("Configure your notification uplinks below to receive real-time trade signals.")
    
    # This tucks the technical IDs away just like a 'Settings' menu in Bitsgap or Pionex
    with st.expander("🔔 Notification Settings", expanded=False):
        st.write("Enter your credentials to enable multi-channel alerts.")
        email_user = st.text_input("Personal Email", placeholder="email@example.com")
        
        st.divider()
        st.markdown("**Telegram Integration**")
        tg_token = st.text_input("Bot Token", type="password", help="Get this from @BotFather on Telegram")
        tg_id = st.text_input("Chat ID", help="Get this from @userinfobot on Telegram")
    
    st.divider()
    
    # Adding a health check button adds to the "Vibe" of a professional OS
    if st.button("System Health Check", use_container_width=True):
        st.success("Quantora Core: Operational")
        st.toast("All systems synchronized.")
        
    st.caption("Quantora Pro Engine v2.1 | © 2026")
# Main Input
col_input, _ = st.columns([2, 1])
with col_input:
    symbol = st.text_input("Select Asset Ticker:", "BTC").upper()

if st.button("RUN QUANT ANALYSIS", use_container_width=True):
    with st.spinner("Synchronizing with Market Nodes..."):
        try:
            # 1. Fetching (Using Daily for Stocks/Crypto)
            url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}'
            data = requests.get(url).json()
            df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index').astype(float).sort_index().tail(60)

            # 2. Indicators
            df['MA10'] = df['4. close'].rolling(window=10).mean()
            df['MA30'] = df['4. close'].rolling(window=30).mean()
            current_price = df['4. close'].iloc[-1]
            signal = "BUY" if df['MA10'].iloc[-1] > df['MA30'].iloc[-1] else "SELL"

            # 3. Professional Charting
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=df.index, y=df['4. close'], name='Price', line=dict(color='#0066FF', width=3)))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA10'], name='10D MA', line=dict(color='#00D084', dash='dot')))
            fig.add_trace(go.Scatter(x=df.index, y=df['MA30'], name='30D MA', line=dict(color='#FF4B4B')))
            fig.update_layout(
                plot_bgcolor='white', 
                paper_bgcolor='rgba(0,0,0,0)',
                margin=dict(l=20, r=20, t=20, b=20),
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
            )
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F0F2F6')
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F0F2F6')
            st.plotly_chart(fig, use_container_width=True)

            # 4. Metrics Cards
            c1, c2, c3 = st.columns(3)
            c1.metric("Market Price", f"${current_price:,.2f}")
            c2.metric("Bot Signal", signal, delta=signal)
            c3.metric("Status", "Active")

            # 5. Multichannel Notifications
            alert_msg = f"🏹 Quantora Pro Alert\nAsset: {symbol}\nSignal: {signal}\nPrice: ${current_price}"
            send_telegram(alert_msg, tg_token, tg_id)
            send_email(alert_msg, email_user)
            st.toast("Alerts Dispatched via Email & Telegram!")

           # 6. AI Insight (Updated for CrewAI Validation)
            my_ai_brain = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
            
            analyst = Agent(
                role='Senior Crypto Strategist', 
                goal=f'Provide a professional market summary for {symbol}', 
                backstory='You are a world-class financial analyst with 20 years of experience in quantitative trading and crypto markets. You are known for concise, high-impact insights.',
                llm=my_ai_brain
            )
            
            task = Task(
                description=f"Analyze the {signal} signal for {symbol} at the price of {current_price}. Explain the significance of the 10-day and 30-day moving average crossover in two professional sentences.", 
                expected_output="A concise 2-sentence market briefing.", 
                agent=analyst
            )
            
            report = Crew(agents=[analyst], tasks=[task]).kickoff()
            
            st.subheader("💡 Strategist Intelligence")
            st.info(report.raw)

        except Exception as e:
            st.error(f"Analysis Failed: {e}")

Overwriting Quantora.py
